# 04 · SARIMA and naive baselines on the notebook-02 output

Same data, blocks, folds, origins and metrics as notebook 03, so the results line up. Per device
and target, a SARIMAX with a daily seasonal part (s = 24) and the weekly cycle as Fourier terms
is fitted on the FIT_DAYS before each fold and re-filtered with the same parameters at every
origin. One order per target is chosen on the tuning folds, refitted on the test folds, then
fitted before CUTOFF, frozen and replayed every 6 h. The naive 168 h and 24 h baselines are
scored as models of their own, with intervals from their empirical errors. Every setting lives
in the first code cell, tagged `parameters` for papermill; the last cell runs it on the GX10.

In [ ]:
# What to fit
TARGETS = ["grid_export", "grid_import"]   # grid_export only runs on devices with PV
HORIZON = 168                              # every origin forecasts hours 1..HORIZON: 48 | 96 | 168
DEVICES = None                             # None: every device; or a list of ids
# Weather is read at the target hour, so the results are an upper bound until as-issued
# weather forecasts land in notebook 01.
WEATHER = "perfect"                        # the only value for now
WEATHER_EXOG = []                          # extra regressors, e.g. ["global_tilted_irradiance"]

# Blocks, local dates (Europe/Rome): tune [TUNE_START, TEST_START), test [TEST_START, CUTOFF),
# then a live replay of the model frozen at CUTOFF
TUNE_START = "2026-06-01"
TEST_START = "2026-08-01"
CUTOFF = "2026-09-01"
TUNE_FOLD_DAYS = 14                        # each fold refits on the FIT_DAYS before it
TEST_FOLD_DAYS = 7
ORIGIN_STRIDE_H = 24                       # tune/test origins from local midnight
REPLAY_STRIDE_H = 6                        # replay origins at local 00/06/12/18 from CUTOFF

# SARIMAX: ((p, d, q), (P, D, Q), K), seasonal period 24, K sin/cos pairs of the weekly cycle.
# One order per target is chosen on the tuning folds, shared by every device.
ORDERS = [((1, 0, 1), (0, 1, 1), 0), ((1, 0, 1), (1, 1, 1), 0), ((1, 0, 1), (0, 1, 1), 2),
          ((1, 0, 1), (1, 1, 0), 2), ((1, 0, 1), (1, 1, 1), 2), ((2, 0, 1), (0, 1, 1), 2),
          ((2, 0, 2), (1, 1, 1), 2), ((1, 0, 1), (0, 1, 1), 4)]
FIT_DAYS = 42                              # each fit, and each re-filter, sees this many days
MAXITER = 50                               # L-BFGS iterations per fit (statsmodels default)
QUANTILES = [0.1, 0.25, 0.5, 0.75, 0.9]   # SARIMA: Gaussian; naive: empirical errors
N_JOBS = None                              # worker processes; None: os.cpu_count()

# Output
RUN_NAME = None                            # None: "sarima-h<HORIZON>-<timestamp>"
LOG_MLFLOW = True                          # MLFLOW_TRACKING_URI if set, else runs/mlflow.db
MLFLOW_EXPERIMENT = "nb04-sarima"

In [ ]:
import json
import multiprocessing
import os
import time
import warnings
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import norm
from statsmodels.tsa.statespace.sarimax import SARIMAX
from threadpoolctl import threadpool_limits

PARAMS = {k: v for k, v in dict(globals()).items() if k.isupper() and not callable(v)}
assert WEATHER == "perfect", "only perfect-forecast weather is available so far"
assert {0.1, 0.25, 0.75, 0.9} <= set(QUANTILES), "the 50% and 80% intervals need these"
assert HORIZON <= 168, "naive168 would read labels after the origin"
N_JOBS = N_JOBS or os.cpu_count()

NB_DIR = Path.cwd().resolve()
if not (NB_DIR / "data").exists() and (NB_DIR / "notebooks" / "data").exists():
    NB_DIR = NB_DIR / "notebooks"
REPO_ROOT = next(p for p in [NB_DIR, *NB_DIR.parents] if (p / "pyproject.toml").exists())
DATA_DIR = NB_DIR / "data"
RUN_NAME = RUN_NAME or f"sarima-h{HORIZON}-{time.strftime('%Y%m%d-%H%M%S')}"
OUT_DIR = REPO_ROOT / "runs" / "nb04" / RUN_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

PT = pd.read_parquet(DATA_DIR / "processed_hourly.parquet")
quality = pd.read_csv(DATA_DIR / "device_quality.csv").set_index("device_id")
HAS_PV = quality["has_pv"].astype(bool).to_dict()
ALL_DEVICES = sorted(DEVICES or PT["device_id"].unique())
PT = PT[PT["device_id"].isin(ALL_DEVICES)]
# One continuous hourly frame per device, so a lag is a plain index lookup.
DEV = {d: g.drop_duplicates("ts_hour", keep="last").set_index("ts_hour").sort_index().asfreq("h")
       for d, g in PT.groupby("device_id")}
LOCAL_TZ = "Europe/Rome"
# Site weather, one value per hour for every device, gaps interpolated (SARIMAX refuses NaN exog).
WX = PT.groupby("ts_hour")[WEATHER_EXOG].first().asfreq("h").interpolate(limit_direction="both")
print("run:", RUN_NAME, "| devices:", len(ALL_DEVICES), "| rows:", len(PT), "| output:", OUT_DIR)

In [ ]:
# Origins as UTC timestamps. ts_hour labels the start of the hour: a forecast issued at clock
# time `origin` has seen labels up to origin - 1h and returns labels origin .. origin +
# (HORIZON - 1)h, lead h = 1..HORIZON. A path counts only if its last label is before the
# block end (tune/test) or has been observed (replay).
T_TUNE, T_TEST, CUT = (pd.Timestamp(d, tz=LOCAL_TZ) for d in (TUNE_START, TEST_START, CUTOFF))
PATH_H = pd.Timedelta(hours=HORIZON)
LEADS = pd.to_timedelta(np.arange(HORIZON), unit="h")  # label offsets from the origin
rows = []
for block, start, end, days in [("tune", T_TUNE, T_TEST, TUNE_FOLD_DAYS),
                                ("test", T_TEST, CUT, TEST_FOLD_DAYS)]:
    for fold in pd.date_range(start, end, freq=f"{days}D", inclusive="left"):
        stop = min(fold + pd.Timedelta(days=days), end)
        rows += [(block, fold, o) for o in pd.date_range(fold, stop, freq=f"{ORIGIN_STRIDE_H}h",
                                                         inclusive="left") if o + PATH_H <= end]
ORIGINS = pd.DataFrame(rows, columns=["block", "fold", "origin"])
ORIGINS[["fold", "origin"]] = ORIGINS[["fold", "origin"]].apply(lambda c: c.dt.tz_convert("UTC"))
T_TUNE, T_TEST, CUT = (t.tz_convert("UTC") for t in (T_TUNE, T_TEST, CUT))
BLOCK_END = {"tune": T_TEST, "test": CUT}
PAIRS = [(d, t) for t in TARGETS for d in ALL_DEVICES if t != "grid_export" or HAS_PV.get(d, False)]

# Every scored row: its actual, the two naive forecasts and is_daylight, keyed by
# (device, target, origin, horizon); rows without an actual are dropped.
parts = []
for d, target in PAIRS:
    s = DEV[d][target]
    last = DEV[d].index[-1] - PATH_H + pd.Timedelta(hours=1)
    replay = pd.date_range(CUT, last, freq=f"{REPLAY_STRIDE_H}h")
    orig = pd.concat([ORIGINS, pd.DataFrame({"block": "replay", "fold": CUT, "origin": replay})],
                     ignore_index=True)
    h = np.tile(np.arange(1, HORIZON + 1), len(orig))
    o = pd.DatetimeIndex(orig["origin"]).repeat(HORIZON)
    t = o + pd.to_timedelta(h - 1, unit="h")
    lag168 = t - pd.Timedelta(hours=168)
    lag24 = t - pd.to_timedelta(24 * np.ceil(h / 24), unit="h")  # most recent observable day
    meta = pd.DataFrame({
        "device_id": d, "target": target, "block": orig["block"].repeat(HORIZON).array,
        "fold": orig["fold"].repeat(HORIZON).array, "origin": o, "horizon": h, "ts_hour": t,
        "actual": s.reindex(t).to_numpy(), "naive168": s.reindex(lag168).to_numpy(),
        "naive24": s.reindex(lag24).to_numpy(),
        "is_daylight": DEV[d]["is_daylight"].reindex(t).to_numpy()})
    assert (meta.groupby("origin").size() == HORIZON).all()  # full paths, before dropping gaps
    assert (lag168 < o).all() and (lag24 < o).all()  # the naives read before the origin
    assert ((replay - CUT) % pd.Timedelta(hours=REPLAY_STRIDE_H) == pd.Timedelta(0)).all()
    parts.append(meta[meta["actual"].notna()])
META = pd.concat(parts, ignore_index=True)
for block, end in BLOCK_END.items():  # no tuning or test actual at or after its block end
    assert (META.loc[META["block"] == block, "ts_hour"] < end).all()
first = META.loc[META["block"] == "replay", "ts_hour"].min()
assert first == CUT, "the first replay target label is CUTOFF"
# Each (device, target, block, fold) is one fit, forecasting from each of its origins.
FOLD_ORIGINS = META.groupby(["device_id", "target", "block", "fold"])["origin"].unique()
print(f"scored rows {len(META):,} | tune/test origins {ORIGINS.groupby('block').size().to_dict()}"
      f" | replay from {first.tz_convert(LOCAL_TZ)}")

In [ ]:
# One job = one SARIMAX fit on the FIT_DAYS of labels before `fit_end`, then, at each origin, a
# re-filter with the same parameters on the FIT_DAYS before it (labels < origin) and a HORIZON
# forecast with the future exog. The Kalman filter skips missing labels. Workers are forked
# processes (statsmodels holds the GIL) that see DEV and WX as they are.
FIT = pd.Timedelta(days=FIT_DAYS)
QCOLS = [f"q{q}" for q in QUANTILES]
Z = norm.ppf(QUANTILES)


def exog(t, k):
    """K sin/cos pairs of the local hour of the week, then WEATHER_EXOG at t; None if empty."""
    local = t.tz_convert(LOCAL_TZ)
    week = 2 * np.pi * (local.dayofweek * 24 + local.hour).to_numpy() / 168
    cols = [f(j * week) for j in range(1, k + 1) for f in (np.sin, np.cos)]
    cols += [WX[c].reindex(t).to_numpy() for c in WEATHER_EXOG]
    return np.column_stack(cols) if cols else None


def sarima_job(d, target, spec, fit_end, origins):
    """Fit ORDERS[spec] for one device and target; (info, forecasts) where forecasts is an
    (origins, HORIZON, 1 + quantiles) array of point then sorted quantiles, clipped at 0,
    all NaN if the fit failed (info["error"])."""
    (order, seasonal, k), s = ORDERS[spec], DEV[d][target]
    info = {"device_id": d, "target": target, "spec": spec, "fold": fit_end, "error": None,
            "converged": np.nan, "fit_s": np.nan, "apply_s": np.nan, "params": None}
    out = np.full((len(origins), HORIZON, 1 + len(QUANTILES)), np.nan)

    def window(end):
        t = pd.date_range(end - FIT, end, freq="h", inclusive="left")
        assert t[-1] < end  # nothing at or after the origin
        return s.reindex(t).to_numpy(), exog(t, k)

    warnings.simplefilter("ignore")  # convergence chatter; converged is counted instead
    with threadpool_limits(1):  # tiny matrices: more BLAS threads only fight each other
        try:
            y, X = window(fit_end)
            if np.isfinite(y).sum() < 7 * 24:
                raise ValueError("under 7 days of observations")
            t0 = time.time()
            res = SARIMAX(y, exog=X, order=tuple(order),
                          seasonal_order=(*seasonal, 24)).fit(disp=False, maxiter=MAXITER)
            info.update(fit_s=time.time() - t0, converged=res.mle_retvals.get("converged", True),
                        params=dict(zip(
                            res.model.param_names, res.params.tolist(), strict=True)))
            t0 = time.time()
            for i, o in enumerate(pd.DatetimeIndex(origins)):
                y, X = window(o)
                fc = res.apply(y, exog=X).get_forecast(HORIZON, exog=exog(o + LEADS, k))
                mean, se = fc.predicted_mean, fc.se_mean
                out[i] = np.column_stack([mean, mean[:, None] + se[:, None] * Z])
            info["apply_s"] = (time.time() - t0) / len(origins)
        except Exception as e:  # e.g. a flat or too short series
            info["error"] = f"{type(e).__name__}: {e}"
    out[..., 1:] = np.sort(out[..., 1:], axis=-1)
    return info, np.maximum(0, out)


def run(jobs):
    """sarima_job over (device, target, spec, fit_end, origins) tuples, N_JOBS at a time."""
    t0, results = time.time(), []
    with ProcessPoolExecutor(N_JOBS, mp_context=multiprocessing.get_context("fork")) as pool:
        for i, r in enumerate(pool.map(sarima_job, *zip(*jobs, strict=True)), 1):
            results.append(r)
            if i % max(1, len(jobs) // 10) == 0 or i == len(jobs):
                print(f"  {i}/{len(jobs)} fits [{time.time() - t0:.0f}s]", flush=True)
    for info, _ in results:
        if info["error"]:
            print(f"WARNING: {info['device_id']} {info['target']} {ORDERS[info['spec']]}"
                  f" fold {info['fold']:%Y-%m-%d}: {info['error']}")
    return results

In [ ]:
# Order search on the tuning folds, one order per target shared by every device. The score of
# a fit is the mean of its MAE over leads 1-48, 1-96 and 1-HORIZON (so each horizon counts
# equally); a candidate scores the mean over folds of the mean over devices. A device whose
# fit fails is left out of that fold, with a warning, and counted.
LEAD_CUTS = sorted({c for c in (48, 96, HORIZON) if c <= HORIZON})


def lead_score(d, target, origins, forecasts):
    """The lead-band MAE of one fit's point forecasts."""
    t = pd.DatetimeIndex(origins).repeat(HORIZON) + np.tile(LEADS, len(origins))
    err = np.abs(forecasts[..., 0].ravel() - DEV[d][target].reindex(t).to_numpy())
    lead = np.tile(np.arange(1, HORIZON + 1), len(origins))
    return np.mean([np.nanmean(err[lead <= c]) for c in LEAD_CUTS])


t0 = time.time()
tune = FOLD_ORIGINS.xs("tune", level="block")
jobs = [(d, t, spec, fold, o) for spec in range(len(ORDERS)) for (d, t, fold), o in tune.items()]
print(f"order search: {len(jobs)} fits")
LOG = []  # one row per fit, every block
for (d, target, _, _, origins), (info, fc) in zip(jobs, run(jobs), strict=True):
    info["score"] = np.nan if info["error"] else lead_score(d, target, origins, fc)
    LOG.append(dict(info, block="tune"))
tl = pd.DataFrame(LOG)
tl["failed"] = tl["error"].notna()
score = tl.groupby(["target", "spec", "fold"])["score"].mean().groupby(["target", "spec"]).mean()
ORDER_TABLE = tl.groupby(["target", "spec"]).agg(
    fits=("failed", "size"), failures=("failed", "sum"),
    not_converged=("converged", lambda c: int((c == False).sum())),  # noqa: E712
    fit_s=("fit_s", "mean"), apply_s=("apply_s", "mean")).assign(score=score)
ORDER_TABLE.insert(0, "order", [ORDERS[s] for _, s in ORDER_TABLE.index])
ORDER_TABLE.to_csv(OUT_DIR / "orders.csv")
BEST = {t: int(ORDER_TABLE.loc[t, "score"].idxmin()) for t in TARGETS}
(OUT_DIR / "best_order.json").write_text(json.dumps(
    {t: {"order": ORDERS[s][0], "seasonal_order": [*ORDERS[s][1], 24], "fourier_k": ORDERS[s][2],
         "weather_exog": WEATHER_EXOG, "score": ORDER_TABLE.loc[(t, s), "score"]}
     for t, s in BEST.items()}, indent=2))
TUNE_SECONDS = time.time() - t0
print(f"tuning folds {sorted(f'{f.tz_convert(LOCAL_TZ):%Y-%m-%d}' for f in tl['fold'].unique())}"
      f" in {TUNE_SECONDS:.0f}s")
display(ORDER_TABLE.round(4))
print({t: ORDERS[s] for t, s in BEST.items()})

In [ ]:
# Test folds refit the chosen order at each fold start; the final fit (FIT_DAYS before CUTOFF)
# is frozen and replayed from CUTOFF. The naive intervals are the point plus the empirical
# quantiles of that baseline's signed error per target and lead hour, pooled over devices, from
# the rows before the block: tune rows for test, tune and test rows for the replay.
t0 = time.time()
evals = FOLD_ORIGINS.drop("tune", level="block")
jobs = [(d, t, BEST[t], fold, o) for (d, t, _, fold), o in evals.items()]
print(f"test + replay: {len(jobs)} fits")
parts, FINAL = [], {}
for (d, target, _, fold, origins), (info, fc) in zip(jobs, run(jobs), strict=True):
    block = "replay" if fold == CUT else "test"
    LOG.append(dict(info, block=block))
    if block == "replay" and info["params"]:
        FINAL[d, target] = info["params"]
    parts.append(pd.DataFrame({"device_id": d, "target": target,
                               "origin": pd.DatetimeIndex(origins).repeat(HORIZON),
                               "horizon": np.tile(np.arange(1, HORIZON + 1), len(origins)),
                               **dict(zip(["point", *QCOLS], fc.reshape(-1, fc.shape[-1]).T,
                                          strict=True))}))
KEY = ["device_id", "target", "origin", "horizon"]
# Only rows SARIMA forecast (its failures drop out of every model), so all three share rows.
EV = META[META["block"] != "tune"].merge(pd.concat(parts), on=KEY).dropna(subset=["point"])
EVAL_SECONDS = time.time() - t0

MODELS = {"sarima": EV}
for name in ["naive168", "naive24"]:
    blocks = []
    for block, before in [("test", ["tune"]), ("replay", ["tune", "test"])]:
        seen, ev = META[META["block"].isin(before)], EV[EV["block"] == block]
        eq = seen.assign(e=seen["actual"] - seen[name]).groupby(["target", "horizon"])[
            "e"].quantile(QUANTILES).unstack()
        eq = eq.reindex(pd.MultiIndex.from_frame(ev[["target", "horizon"]])).to_numpy()
        q = ev[name].to_numpy()[:, None] + eq
        blocks.append(ev.assign(point=ev[name], **dict(zip(QCOLS, np.sort(np.maximum(
            0, q), axis=1).T, strict=True))))
    MODELS[name] = pd.concat(blocks).dropna(subset=["point"])
RES = pd.concat([m.assign(model=name) for name, m in MODELS.items()], ignore_index=True)

(OUT_DIR / "models").mkdir(exist_ok=True)
for (d, target), params in FINAL.items():
    order, seasonal, k = ORDERS[BEST[target]]
    (OUT_DIR / "models" / f"{d}_{target}_sarima.json").write_text(json.dumps({
        "order": order, "seasonal_order": [*seasonal, 24], "fourier_k": k,
        "exog": [f"{f}{j}_hour_of_week_local" for j in range(1, k + 1) for f in ("sin", "cos")]
        + WEATHER_EXOG, "fit_days": FIT_DAYS, "fit_end": str(CUT),
        "params": params}, indent=2))
COLS = ["model", "scope", "device_id", "target", "origin", "horizon", "ts_hour", "actual", "point",
        *QCOLS]
out = RES.assign(scope="per_device")
out[out["block"] == "test"][[*COLS[:4], "fold", *COLS[4:]]].to_parquet(
    OUT_DIR / "test.parquet", index=False)
out[out["block"] == "replay"][COLS].to_parquet(OUT_DIR / "replay.parquet", index=False)
LOG = pd.DataFrame(LOG).drop(columns="params")
LOG.to_csv(OUT_DIR / "fits.csv", index=False)
print(f"{len(LOG)} fits: {LOG['fit_s'].mean():.1f}s per fit, {LOG['apply_s'].mean():.2f}s per"
      f" origin (re-filter + forecast) | {LOG['error'].notna().sum()} failed,"
      f" {(LOG['converged'] == False).sum()} not converged")  # noqa: E712

In [ ]:
BANDS = ["1-8", "9-24", "25-48", "49-96", "97-168", "all"]
QA = np.array(QUANTILES)
RES["band"] = pd.cut(RES["horizon"], [0, 8, 24, 48, 96, 168], labels=BANDS[:-1]).astype(str)
RES["export_day"] = RES["is_daylight"].eq(1) & RES["target"].eq("grid_export")


def metrics(g):
    """Point and interval scores of a group of rows, and skill vs naive168, in kWh/h."""
    y, err = g["actual"], g["point"] - g["actual"]
    diff = y.to_numpy()[:, None] - g[QCOLS].to_numpy()
    m = {"mae": err.abs().mean(), "rmse": np.sqrt((err**2).mean()),
         "mae_naive168": (g["naive168"] - y).abs().mean()}
    m["skill168"] = 1 - m["mae"] / m["mae_naive168"] if m["mae_naive168"] > 0.01 else np.nan
    m["pinball"] = np.maximum(QA * diff, (QA - 1) * diff).mean()
    m["cov50"] = y.between(g["q0.25"], g["q0.75"]).mean()
    m["cov80"] = y.between(g["q0.1"], g["q0.9"]).mean()
    m["mae_daylight"] = err[g["export_day"]].abs().mean()  # export only
    return pd.Series(m)


both = pd.concat([RES, RES.assign(band="all")])
both["band"] = pd.Categorical(both["band"], BANDS)
SUMMARY = both.groupby(["block", "model", "target", "band"], observed=True).apply(
    metrics, include_groups=False)
PER_DEVICE = RES.groupby(["block", "model", "target", "device_id"]).apply(
    metrics, include_groups=False)
PER_DEVICE = PER_DEVICE[["mae", "mae_naive168", "skill168"]].unstack("block")
SUMMARY.to_csv(OUT_DIR / "summary.csv")
PER_DEVICE.to_csv(OUT_DIR / "per_device.csv")
print(f"tuned in {TUNE_SECONDS:.0f}s, test + replay in {EVAL_SECONDS:.0f}s | kWh/h")
display(SUMMARY.round(4))
display(PER_DEVICE.round(4))

In [ ]:
COLORS = {"sarima": "#3b6fb6", "naive168": "#999999", "naive24": "#d9822b"}
fig, axes = plt.subplots(len(TARGETS), 2, figsize=(16, 4.5 * len(TARGETS)), squeeze=False)
for row, target in zip(axes, TARGETS, strict=True):
    one = RES[RES["target"] == target]
    lead = one.assign(ae=(one["point"] - one["actual"]).abs()).groupby(
        ["block", "model", "horizon"])["ae"].mean()
    for (block, model), s in lead.groupby(level=["block", "model"]):
        s.droplevel([0, 1]).plot(ax=row[0], color=COLORS[model], lw=2 if block == "test" else 1,
                                 ls="-" if block == "test" else "--", label=f"{model} {block}")
    row[0].set(title=f"{target}: MAE by lead hour", xlabel="lead (h)", ylabel="kWh/h")
    row[0].legend()
    path = one[(one["block"] == "replay") & (one["model"] == "sarima")]
    path = path[path["device_id"] == path["device_id"].iloc[0]]
    path = path[path["origin"] == path["origin"].min()]
    row[1].fill_between(path["ts_hour"], path["q0.1"], path["q0.9"], color=COLORS["sarima"],
                        alpha=0.2, label="10-90%")
    row[1].plot(path["ts_hour"], path["actual"], color="#222222", lw=2, label="actual")
    row[1].plot(path["ts_hour"], path["point"], color=COLORS["sarima"], lw=2, label="SARIMA")
    start = path["origin"].iloc[0].tz_convert(LOCAL_TZ)
    row[1].set(title=f"{target}: replay, {path['device_id'].iloc[0]} from {start:%Y-%m-%d %H:%M}"
               " local", ylabel="kWh/h")
    row[1].tick_params(axis="x", rotation=30)
    row[1].legend()
fig.tight_layout()
fig.savefig(OUT_DIR / "report.png", dpi=110)

In [ ]:
if LOG_MLFLOW:
    import mlflow

    default_uri = f"sqlite:///{REPO_ROOT / 'runs' / 'mlflow.db'}"
    mlflow.set_tracking_uri(os.environ.get("MLFLOW_TRACKING_URI", default_uri))
    mlflow.set_experiment(MLFLOW_EXPERIMENT)
    with mlflow.start_run(run_name=RUN_NAME):
        mlflow.log_params({**PARAMS, "host": os.uname().nodename,
                           "tune_seconds": round(TUNE_SECONDS), "eval_seconds": round(EVAL_SECONDS),
                           **{f"best.{t}": ORDERS[s] for t, s in BEST.items()}})
        mlflow.log_metrics({f"{block}.{model}.{metric}.{t}.{band}": v
                            for (block, model, t, band), r in SUMMARY.iterrows()
                            for metric, v in r.items() if pd.notna(v)})
        mlflow.log_artifacts(str(OUT_DIR))
    print("logged to", mlflow.get_tracking_uri())

In [ ]:
# Run this notebook on the GX10 instead of here: save the notebook, run the setup cell (the
# third one) and this one, then call gx10_run() below; keyword arguments override the top
# cell, e.g. gx10_run(N_JOBS=18). Jobs can run side by side; gx10_status() shows them,
# gx10_pull() brings runs/ and the MLflow runs back.
# The job runs in a Docker container with a memory cap, because the box freezes if it runs
# out of memory (the Qwen server already holds about 65 GB). It keeps going if the laptop
# disconnects; results and the MLflow db land in runs/ on the box.
import shlex
import subprocess

GX10_HOST = os.environ.get("GX10_HOST")                # user@host of the box, set it in your shell
GX10_DIR = "celine-forecasting"                     # on the box, relative to the home folder
GX10_MEMORY, GX10_CPUS = "24g", 18                  # the box has 20 cores; N_JOBS within this
GX10_IMAGE = "ghcr.io/astral-sh/uv:python3.12-bookworm"


def gx10_run(**overrides):
    """Copy the repo and 02's outputs to the box, then start this notebook there, with
    `overrides` (parameter name: value) on top of the top cell."""
    if not GX10_HOST:
        raise RuntimeError("set GX10_HOST (user@host) in the environment before using the GX10 helpers")
    p = {**PARAMS, **overrides}
    job = f"nb04-sarima-h{p['HORIZON']}-{time.strftime('%Y%m%d-%H%M%S')}"
    files = subprocess.run(["git", "ls-files", "-co", "--exclude-standard"], cwd=REPO_ROOT,
                           check=True, capture_output=True, text=True).stdout
    subprocess.run(["rsync", "-az", "--files-from=-", "./", f"{GX10_HOST}:{GX10_DIR}/"],
                   input=files, cwd=REPO_ROOT, check=True, text=True)
    inputs = [str(DATA_DIR / f) for f in ["processed_hourly.parquet", "device_quality.csv"]]
    subprocess.run(["rsync", "-az", *inputs, f"{GX10_HOST}:{GX10_DIR}/notebooks/data/"], check=True)
    params = json.dumps(p)
    remote = (
        # only docker goes to the background
        f"cd {GX10_DIR} && mkdir -p runs/jobs/{job} || exit 1; "
        f"nohup docker run --rm --name {job} --memory {GX10_MEMORY} --cpus {GX10_CPUS} "
        "--user $(id -u):$(id -g) -v $PWD:/w -w /w -e HOME=/tmp "
        "-e UV_CACHE_DIR=/w/.uv-cache -e UV_PROJECT_ENVIRONMENT=/w/.venv-docker "
        f"{GX10_IMAGE} uv run --frozen --extra notebooks --extra mlflow papermill "
        f"notebooks/04_sarima.ipynb runs/jobs/{job}/04_sarima.ipynb --cwd notebooks "
        f"-y {shlex.quote(params)} > runs/jobs/{job}/log.txt 2>&1 < /dev/null &")
    subprocess.run(["ssh", GX10_HOST, remote], check=True)
    print(f"started {job} on {GX10_HOST}")


def gx10_status():
    """Each running nb04 job with the last lines of its log, then the latest nb04 job folder."""
    remote = (f"cd {GX10_DIR}; for n in $(docker ps --filter name=nb04- --format '{{{{.Names}}}}');"
              " do docker ps --filter name=$n --format '{{.Names}}  {{.Status}}'; "
              "tail -c 600 runs/jobs/$n/log.txt | tr '\r' '\n' | tail -n 3; done; "
              "echo latest: $(ls -td runs/jobs/nb04-*/ | head -1)")
    print(subprocess.run(["ssh", GX10_HOST, remote], capture_output=True, text=True).stdout)


def gx10_pull():
    """Copy runs/ from the box to runs/gx10/ here, then its MLflow runs to our tracking server."""
    local = REPO_ROOT / "runs" / "gx10"
    subprocess.run(["rsync", "-az", f"{GX10_HOST}:{GX10_DIR}/runs/", str(local)], check=True)
    gx10_to_mlflow(local / "mlflow.db")


def gx10_to_mlflow(db):
    """Copy the box's runs (params, metrics, tags) to MLflow here, skipping those already copied."""
    from mlflow.entities import Metric, Param
    from mlflow.tracking import MlflowClient

    src = MlflowClient(f"sqlite:///{db}")
    uri = os.environ.get("MLFLOW_TRACKING_URI", f"sqlite:///{REPO_ROOT / 'runs' / 'mlflow.db'}")
    dst = MlflowClient(uri)
    copied = 0
    for exp in src.search_experiments():
        target = dst.get_experiment_by_name(exp.name)
        exp_id = target.experiment_id if target else dst.create_experiment(exp.name)
        # The box's run id travels as a tag, so pulling twice never copies a run twice.
        done = {r.data.tags.get("gx10.run_id")
                for r in dst.search_runs([exp_id], max_results=50000)}
        for run in src.search_runs([exp.experiment_id], max_results=50000):
            if run.info.run_id in done:
                continue
            tags = {**run.data.tags, "gx10.run_id": run.info.run_id, "gx10.host": GX10_HOST}
            new = dst.create_run(exp_id, start_time=run.info.start_time, tags=tags,
                                 run_name=run.info.run_name).info.run_id
            metrics = [Metric(m.key, m.value, m.timestamp, m.step) for k in run.data.metrics
                       for m in src.get_metric_history(run.info.run_id, k)]
            params = [Param(k, v) for k, v in run.data.params.items()]
            for i in range(0, len(metrics), 1000):  # log_batch caps: 1000 metrics, 100 params
                dst.log_batch(new, metrics=metrics[i:i + 1000])
            for i in range(0, len(params), 100):
                dst.log_batch(new, params=params[i:i + 100])
            dst.set_terminated(new, run.info.status, run.info.end_time)
            copied += 1
    print(f"copied {copied} new run(s) from {db} to {uri}")

In [ ]:
# gx10_run(N_JOBS=18)
# gx10_status()
# gx10_pull()